In [ ]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import os

# Load the cleaned datasets
df_alcohol = pd.read_csv('../data/processed/alcohol_recoded.csv')
df_bmi = pd.read_csv('../data/processed/bmi_cleaned.csv')

In [ ]:
print("=== Inference 1: Population Proportion (CurrentAlcoholUse) ===")

# 1. Calculate sample proportion
n_prop = len(df_alcohol)
successes = df_alcohol['Alcohol_Binary'].sum()
p_hat = successes / n_prop

# 2. Set benchmark
p_0 = 0.35 

# 3. Construct 95% Confidence Interval
ci_low_prop, ci_up_prop = proportion_confint(count=successes, nobs=n_prop, alpha=0.05, method='normal')

# 4. Conduct one-sample Z-test
stat_prop, pval_prop = proportions_ztest(count=successes, nobs=n_prop, value=p_0, prop_var=p_0)

print(f"Sample Proportion: {p_hat:.4f}")
print(f"Benchmark (p0): {p_0}")
print(f"95% Confidence Interval: ({ci_low_prop:.4f}, {ci_up_prop:.4f})")
print(f"P-value: {pval_prop:.4e}")

### Interpretation for Proportion Analysis
* **What was estimated/tested:** We estimated the population proportion of students who currently use alcohol and tested if it differs from the benchmark of 0.35.
* **Conclusion:** Since the p-value is extremely small (< 0.05), we strongly reject the null hypothesis. There is significant evidence that the true proportion of students using alcohol is higher than 0.35.

In [ ]:
print("\n=== Inference 2: Population Mean (BMIPCT) ===")

# 1. Calculate sample statistics
n_mean = len(df_bmi)
x_bar = df_bmi['BMIPCT'].mean()
s_std = df_bmi['BMIPCT'].std()

# 2. Set benchmark
mu_0 = 65.0 

# 3. Construct 95% Confidence Interval
ci_low_mean, ci_up_mean = stats.t.interval(confidence=0.95, df=n_mean-1, loc=x_bar, scale=s_std/(n_mean**0.5))

# 4. Conduct one-sample t-test
stat_mean, pval_mean = stats.ttest_1samp(a=df_bmi['BMIPCT'], popmean=mu_0)

print(f"Sample Mean: {x_bar:.4f}")
print(f"Benchmark (mu0): {mu_0}")
print(f"95% Confidence Interval: ({ci_low_mean:.4f}, {ci_up_mean:.4f})")
print(f"P-value: {pval_mean:.4f}")

# --- Export Final Summary ---
os.makedirs('../outputs/summary', exist_ok=True)
summary_text = f"""=== Final Inference Summary ===
1. CurrentAlcoholUse (Benchmark p0 = 0.35):
   - Sample Proportion: {p_hat:.4f}
   - 95% CI: ({ci_low_prop:.4f}, {ci_up_prop:.4f})
   - Result: Reject Null Hypothesis (p < 0.05).

2. BMIPCT (Benchmark mu0 = 65.0):
   - Sample Mean: {x_bar:.4f}
   - 95% CI: ({ci_low_mean:.4f}, {ci_up_mean:.4f})
   - Result: Fail to Reject Null Hypothesis (p = 0.4816).
"""
with open('../outputs/summary/final_summary.txt', 'w') as f:
    f.write(summary_text)
    
print("\n✅ Successfully saved summary text to ../outputs/summary/")

### Interpretation for Mean Analysis
* **What was estimated/tested:** We estimated the true population mean of students' BMI percentiles and tested if it differs from the benchmark of 65.0.
* **Conclusion:** The p-value is 0.4816. Since p > 0.05, we fail to reject the null hypothesis. We do not have enough statistical evidence to say the true mean differs significantly from 65.0.

---
### 5. Final Synthesis and Conclusion
In this project, we investigated two key health indicators: alcohol consumption and BMI percentiles among students. 
Our inferential analysis confirmed our EDA findings: student alcohol use (approx. 45.17%) is significantly higher than the 0.35 benchmark, indicating a prevalent behavior deviation. Conversely, the average BMI percentile (64.82) shows no statistically significant difference from the 65.0 benchmark, despite wide individual variation. Overall, alcohol use represents a more significant departure from our expected benchmarks than the BMI percentile.